# Graft/Repair Candidate Generation

This notebook isolates the graft-and-repair candidate-generation process from `generate_feasible_candidates_FAEZ_CLAUDE.py`.

The workflow is organized into three stages:

1. **Load and organize the design data.**
2. **Define the grafting, sanitation, surrogate judging, and repair process.**
3. **Execute graft/repair generation and save a diverse candidate set.**

The interpolation, factorized-sampling, and GMM strategies from the original script are intentionally excluded. The output remains surrogate-judged and must be checked with the real constraint evaluator before it is treated as feasible.

In [1]:
# Import the data, modeling, and distance tools used by the workflow.

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


warnings.filterwarnings("ignore")

# Use a fixed random seed so candidate generation is repeatable.
RNG = np.random.default_rng(7)

# Resolve the paths relative to the project root containing this notebook.
DATA_PATH = Path("Dataset_Structures_V2_1/Combined_Designs_Data.csv")
OUTPUT_PATH = Path("candidate_feasible_graft_designs.csv")

# Preserve the graft/repair settings from the original script.
N_GRAFT = 6000
P_KEEP = 0.50
N_SELECT = 700  # Matches the original script's quota for graft candidates.
PIN_RATIO = 0.50
REPAIR_ITERS = 3

## 1. Load and organize the design data

The source CSV stores the 120 design parameters first, followed by metadata and 25 Boolean constraint results. A design is treated as known-feasible only when all 25 constraint columns pass.

In [3]:
# Load the combined design table and normalize any whitespace in column names.

df = pd.read_csv(DATA_PATH)
print(len(df))
# We only want the first 6020 rows
df = df.iloc[:6020, :]


df.columns = [column.strip() for column in df.columns]

# Preserve the positional column contract used by the original script.
parameter_columns = list(df.columns[:120])
constraint_columns = list(df.columns[123:148])

X = df[parameter_columns].to_numpy(dtype=float)
constraint_pass = df[constraint_columns].to_numpy(dtype=bool)

# Count violations and identify the verified-feasible rows.
num_violations = (~constraint_pass).sum(axis=1)
is_feasible = num_violations == 0
is_generated = (df["Design_Type"] == "Generated").to_numpy()

# Encode ship type as tanker=0, container=1, and bulk carrier=2.
ship_type = np.where(
    df["cat tanker"] == 1,
    0,
    np.where(df["cat container"] == 1, 1, 2),
)

longitudinal_bulkhead = df["longitudinal bulkhead bit"].to_numpy(dtype=int)
num_parameters = len(parameter_columns)
column_index = {name: index for index, name in enumerate(parameter_columns)}

print(f"Loaded {len(df):,} designs with {is_feasible.sum():,} known-feasible designs.")
print(f"Design parameters: {num_parameters}")
print(f"Constraint results: {len(constraint_columns)}")

6120
Loaded 6,020 designs with 0 known-feasible designs.
Design parameters: 120
Constraint results: 25


In [4]:
# Derive parameter metadata used during grafting and sanitation.

integer_columns = np.array(
    [np.allclose(X[:, j], np.round(X[:, j])) for j in range(num_parameters)]
)
binary_columns = np.array(
    [set(np.unique(X[:, j])) <= {0.0, 1.0} for j in range(num_parameters)]
)

category_columns = [
    column_index["cat tanker"],
    column_index["cat container"],
    column_index["cat bulkcarrier"],
]

global_low = X.min(axis=0)
global_high = X.max(axis=0)

# Record parameters that are inapplicable, and therefore always zero,
# for each ship type.
zero_by_ship_type = {}

for type_index in range(3):
    zero_by_ship_type[type_index] = np.array(
        [
            (X[ship_type == type_index, j] == 0).all()
            and not (X[:, j] == 0).all()
            for j in range(num_parameters)
        ]
    )

# Define parameter groups controlled by longitudinal-bulkhead and hatch bits.
LBK_BIT = column_index["longitudinal bulkhead bit"]
LBK_GROUP = [
    column_index[name]
    for name in parameter_columns
    if "long blkhd" in name
] + [column_index["num longitudinal bulkheads"]]

HATCH_BIT = column_index["Hatch Openings Bit"]
HATCH_GROUP = [
    column_index["Hatch Opening L"],
    column_index["Hatch Opening W"],
    column_index["Hatch Opening R"],
]

BLOCKS = [LBK_GROUP + [LBK_BIT], HATCH_GROUP + [HATCH_BIT]]
blocked_columns = {
    index
    for block in BLOCKS
    for index in block
} | set(category_columns)

# A parameter is "pinned" when its range among feasible designs is less
# than PIN_RATIO times its range across the complete dataset.
dataset_range = global_high - global_low
feasible_range = X[is_feasible].max(axis=0) - X[is_feasible].min(axis=0)
range_ratio = np.where(
    dataset_range > 0,
    feasible_range / np.maximum(dataset_range, 1e-12),
    1.0,
)

is_pinned = (range_ratio < PIN_RATIO) & ~binary_columns
is_pinned[category_columns] = False

pinned_indices = np.where(is_pinned)[0]
free_indices = np.array(
    [
        j
        for j in range(num_parameters)
        if not is_pinned[j] and j not in blocked_columns
    ]
)

print(f"Pinned parameters ({len(pinned_indices)}):")
display(pd.Series([parameter_columns[j] for j in pinned_indices], name="parameter"))

ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:
# Build compatible donor pools and the infeasible seed pool.

donors_by_island = {}

for type_index in range(3):
    for bulkhead_bit in (0, 1):
        donors_by_island[(type_index, bulkhead_bit)] = np.where(
            is_feasible
            & (ship_type == type_index)
            & (longitudinal_bulkhead == bulkhead_bit)
        )[0]

# Original dataset designs provide the free-parameter distributions.
dataset_rows_by_ship_type = {
    type_index: np.where((~is_generated) & (ship_type == type_index))[0]
    for type_index in range(3)
}

# Grafting starts from infeasible, non-generated designs.
infeasible_seed_pool = np.where((~is_generated) & (~is_feasible))[0]

# The repair target is the feasible-set 75th percentile for each ship type
# and parameter, calculated over nonzero values.
feasible_q75 = np.zeros((3, num_parameters))

for type_index in range(3):
    feasible_type_values = X[is_feasible & (ship_type == type_index)]

    for j in range(num_parameters):
        nonzero_values = feasible_type_values[:, j][feasible_type_values[:, j] != 0]
        feasible_q75[type_index, j] = (
            np.percentile(nonzero_values, 75)
            if len(nonzero_values)
            else 0.0
        )

donor_summary = pd.DataFrame(
    [
        {
            "ship_type": type_index,
            "longitudinal_bulkhead_bit": bulkhead_bit,
            "num_feasible_donors": len(indices),
        }
        for (type_index, bulkhead_bit), indices in donors_by_island.items()
    ]
)

display(donor_summary)
print(f"Infeasible seed designs: {len(infeasible_seed_pool):,}")

## 2. Define the grafting and repair process

The process trains one calibrated classifier for each constraint. These classifiers act as inexpensive proxy judges. Grafting then combines an infeasible seed's free parameters with pinned parameters from one compatible feasible donor. Finally, the repair loop modifies local variables associated with predicted constraint failures.

In [ ]:
# Train one calibrated pass/fail classifier for each constraint.

print("Training calibrated per-constraint surrogate judges...")

constraint_judges = []
constant_constraint_values = {}

for constraint_index, constraint_name in enumerate(constraint_columns):
    y = constraint_pass[:, constraint_index].astype(int)

    # A constraint with only one observed class does not require a model.
    if y.min() == y.max():
        constraint_judges.append(None)
        constant_constraint_values[constraint_index] = float(y[0])
        continue

    model = CalibratedClassifierCV(
        HistGradientBoostingClassifier(max_iter=150, random_state=0),
        method="isotonic",
        cv=3,
    )

    constraint_judges.append(model.fit(X, y))


def judge_probabilities(candidate_values):
    """Return the predicted pass probability for every candidate/constraint."""

    probabilities = np.ones((len(candidate_values), len(constraint_columns)))

    for constraint_index in range(len(constraint_columns)):
        model = constraint_judges[constraint_index]

        if model is None:
            probabilities[:, constraint_index] = constant_constraint_values[
                constraint_index
            ]
        else:
            probabilities[:, constraint_index] = model.predict_proba(
                candidate_values
            )[:, 1]

    return probabilities


# Establish the surrogate-score band assigned to known-feasible designs.
known_feasible_score = judge_probabilities(X[is_feasible]).prod(axis=1)

print(
    "Known-feasible P(all pass): "
    f"median={np.median(known_feasible_score):.3f}, "
    f"p10={np.percentile(known_feasible_score, 10):.3f}, "
    f"min={known_feasible_score.min():.3f}"
)

In [ ]:
# Optional diagnostic retained from the original script.
# It checks whether out-of-fold surrogate scores distinguish known-feasible
# designs without evaluating each row using a model trained on that row.

oof_probabilities = np.ones((len(df), len(constraint_columns)))
cross_validation = StratifiedKFold(3, shuffle=True, random_state=0)

for constraint_index in range(len(constraint_columns)):
    y = constraint_pass[:, constraint_index].astype(int)

    if y.min() == y.max():
        continue

    for train_indices, test_indices in cross_validation.split(X, y):
        model = HistGradientBoostingClassifier(
            max_iter=150,
            random_state=0,
        ).fit(X[train_indices], y[train_indices])

        oof_probabilities[test_indices, constraint_index] = model.predict_proba(
            X[test_indices]
        )[:, 1]

oof_score = oof_probabilities.prod(axis=1)
top_100 = np.argsort(-oof_score)[:100]

print(
    f"OOF AUC(feasible)={roc_auc_score(is_feasible, oof_score):.4f}; "
    f"known-feasible designs in top 100={is_feasible[top_100].sum()}/"
    f"{is_feasible.sum()}"
)

In [ ]:
# Learn a simple variable-to-constraint repair map from the dataset.
# For each constraint, retain up to three non-pinned, non-binary parameters
# with the strongest positive correlation with passing that constraint.

standardized_X = (X - X.mean(axis=0)) / np.maximum(X.std(axis=0), 1e-12)
repair_map = {}

for constraint_index in range(len(constraint_columns)):
    y = constraint_pass[:, constraint_index].astype(float)

    if y.min() == y.max():
        repair_map[constraint_index] = []
        continue

    standardized_y = (y - y.mean()) / y.std()
    pass_correlation = standardized_X.T @ standardized_y / len(y)

    candidate_levers = [
        parameter_index
        for parameter_index in np.argsort(-pass_correlation)
        if pass_correlation[parameter_index] > 0.05
        and not is_pinned[parameter_index]
        and not binary_columns[parameter_index]
        and parameter_index not in category_columns
    ]

    repair_map[constraint_index] = candidate_levers[:3]

repair_rows = []

for constraint_index, parameter_indices in repair_map.items():
    for rank, parameter_index in enumerate(parameter_indices, start=1):
        repair_rows.append(
            {
                "constraint": constraint_columns[constraint_index],
                "lever_rank": rank,
                "parameter": parameter_columns[parameter_index],
            }
        )

display(pd.DataFrame(repair_rows))

In [ ]:
def jitter_pinned_values(values):
    """Add bounded variation to transplanted donor values."""

    width = np.maximum(
        feasible_range[pinned_indices],
        0.05 * dataset_range[pinned_indices],
    ) * 0.15

    return values + RNG.uniform(
        -1,
        1,
        size=(len(values), len(pinned_indices)),
    ) * width


def sanitize(candidate_values, candidate_ship_types):
    """Restore structural encoding rules after generation or repair."""

    candidate_values = np.clip(candidate_values, global_low, global_high)

    # Restore a valid one-hot ship category and zero inapplicable parameters.
    candidate_values[:, category_columns] = 0

    for type_index in range(3):
        rows = candidate_ship_types == type_index
        candidate_values[np.ix_(rows, [category_columns[type_index]])] = 1
        candidate_values[np.ix_(rows, np.where(zero_by_ship_type[type_index])[0])] = 0

    # Snap all binary columns back to zero or one.
    candidate_values[:, binary_columns] = (
        candidate_values[:, binary_columns] >= 0.5
    ).astype(float)

    # Keep bit-gated parameter groups consistent with their controlling bit.
    for bit_index, group_indices in [
        (LBK_BIT, LBK_GROUP),
        (HATCH_BIT, HATCH_GROUP),
    ]:
        bit_is_off = candidate_values[:, bit_index] < 0.5
        candidate_values[np.ix_(bit_is_off, group_indices)] = 0

        # If a bit is on but its complete parameter group is empty, restore
        # that group from a same-type dataset design with the bit enabled.
        bit_is_on_but_group_is_empty = (
            (~bit_is_off)
            & (np.abs(candidate_values[:, group_indices]).sum(axis=1) < 1e-9)
        )

        for row_index in np.where(bit_is_on_but_group_is_empty)[0]:
            pool = dataset_rows_by_ship_type[int(candidate_ship_types[row_index])]
            enabled_sources = pool[X[pool, bit_index] == 1]
            source_index = RNG.choice(enabled_sources)
            candidate_values[row_index, group_indices] = X[source_index, group_indices]

    # Restore integer-valued parameters after all continuous operations.
    candidate_values[:, integer_columns] = np.round(
        candidate_values[:, integer_columns]
    )

    return candidate_values


def repair(candidate_values, candidate_ship_types):
    """Raise local levers for constraints predicted to fail."""

    for _ in range(REPAIR_ITERS):
        probabilities = judge_probabilities(candidate_values)
        changed = False

        for constraint_index in range(len(constraint_columns)):
            predicted_to_fail = probabilities[:, constraint_index] < 0.60
            lever_indices = repair_map[constraint_index]

            if not predicted_to_fail.any() or not lever_indices:
                continue

            for parameter_index in lever_indices:
                target = feasible_q75[
                    candidate_ship_types[predicted_to_fail],
                    parameter_index,
                ]

                repaired_values = np.maximum(
                    candidate_values[predicted_to_fail, parameter_index],
                    target,
                )

                if not np.allclose(
                    repaired_values,
                    candidate_values[predicted_to_fail, parameter_index],
                ):
                    changed = True

                candidate_values[predicted_to_fail, parameter_index] = repaired_values

        candidate_values = sanitize(candidate_values, candidate_ship_types)

        if not changed:
            break

    return candidate_values

In [ ]:
def generate_graft_candidates(num_candidates):
    """Generate seed/donor hybrids before surrogate-guided repair."""

    seed_indices = RNG.choice(
        infeasible_seed_pool,
        num_candidates,
        replace=len(infeasible_seed_pool) < num_candidates,
    )

    candidate_values = X[seed_indices].copy()
    candidate_ship_types = ship_type[seed_indices].copy()

    # Transplant the complete pinned-parameter block from one compatible,
    # known-feasible donor into each infeasible seed.
    for candidate_index, seed_index in enumerate(seed_indices):
        island = (
            ship_type[seed_index],
            longitudinal_bulkhead[seed_index],
        )
        donor_indices = donors_by_island[island]
        donor_index = RNG.choice(donor_indices)

        candidate_values[candidate_index, pinned_indices] = X[
            donor_index,
            pinned_indices,
        ]

    # Introduce controlled variation so donor parameter blocks are not
    # reproduced as exact point masses.
    candidate_values[:, pinned_indices] = jitter_pinned_values(
        candidate_values[:, pinned_indices]
    )

    candidate_values = sanitize(candidate_values, candidate_ship_types)

    return candidate_values, candidate_ship_types, seed_indices

## 3. Execute graft/repair sample generation

This stage generates the grafted designs, applies surrogate-guided repair, filters on predicted joint feasibility, removes duplicate candidates, and selects a diverse high-scoring subset.

In [ ]:
# Generate the requested number of grafted seed/donor hybrids.

grafted_values, grafted_ship_types, grafted_seed_indices = generate_graft_candidates(
    N_GRAFT
)

# Repair local variables associated with predicted constraint failures.
repaired_values = repair(grafted_values, grafted_ship_types)

# Score each candidate using the product of its 25 calibrated pass
# probabilities, matching the original script's screening criterion.
candidate_constraint_probabilities = judge_probabilities(repaired_values)
candidate_joint_probability = candidate_constraint_probabilities.prod(axis=1)
keep_candidate = candidate_joint_probability >= P_KEEP

accepted_values = repaired_values[keep_candidate]
accepted_ship_types = grafted_ship_types[keep_candidate]
accepted_seed_indices = grafted_seed_indices[keep_candidate]
accepted_constraint_probabilities = candidate_constraint_probabilities[keep_candidate]
accepted_joint_probability = candidate_joint_probability[keep_candidate]

print(
    f"Generated {N_GRAFT:,} graft candidates; "
    f"accepted {keep_candidate.sum():,} with P(all pass) >= {P_KEEP:.2f}."
)
print(f"Median candidate P(all pass): {np.median(candidate_joint_probability):.3f}")
print(f"Candidates with P(all pass) >= 0.90: {(candidate_joint_probability >= 0.90).sum():,}")

In [ ]:
# Remove duplicate accepted candidates while keeping their provenance and
# surrogate scores synchronized.

_, unique_indices = np.unique(
    np.round(accepted_values, 4),
    axis=0,
    return_index=True,
)
unique_indices = np.sort(unique_indices)

accepted_values = accepted_values[unique_indices]
accepted_ship_types = accepted_ship_types[unique_indices]
accepted_seed_indices = accepted_seed_indices[unique_indices]
accepted_constraint_probabilities = accepted_constraint_probabilities[unique_indices]
accepted_joint_probability = accepted_joint_probability[unique_indices]

print(f"Accepted pool after deduplication: {len(accepted_values):,}")

# Standardize candidates using the complete dataset's parameter scales.
parameter_std = np.maximum(X.std(axis=0), 1e-12)
standardized_candidates = (accepted_values - X.mean(axis=0)) / parameter_std


def farthest_point_selection(candidate_indices, num_to_select):
    """Greedily select a spread-out subset, starting from the best score."""

    if len(candidate_indices) <= num_to_select:
        return candidate_indices

    values = standardized_candidates[candidate_indices]
    selected_local_indices = [
        int(np.argmax(accepted_joint_probability[candidate_indices]))
    ]

    minimum_distance = np.linalg.norm(
        values - values[selected_local_indices[0]],
        axis=1,
    )

    for _ in range(num_to_select - 1):
        next_local_index = int(np.argmax(minimum_distance))
        selected_local_indices.append(next_local_index)

        distance_to_new_point = np.linalg.norm(
            values - values[next_local_index],
            axis=1,
        )
        minimum_distance = np.minimum(minimum_distance, distance_to_new_point)

    return candidate_indices[np.array(selected_local_indices)]


# Match the original selection policy: retain the top 60% by score before
# using farthest-point selection to favor parameter-space diversity.
score_cutoff = np.percentile(accepted_joint_probability, 40)
high_scoring_indices = np.where(
    accepted_joint_probability >= score_cutoff
)[0]
selected_indices = farthest_point_selection(high_scoring_indices, N_SELECT)

print(f"Selected {len(selected_indices):,} diverse graft/repair candidates.")

In [ ]:
# Assemble the final table with surrogate scores and graft provenance.

output = pd.DataFrame(
    accepted_values[selected_indices],
    columns=parameter_columns,
)

selected_constraint_probabilities = accepted_constraint_probabilities[selected_indices]
weakest_constraint_indices = np.argmin(
    selected_constraint_probabilities,
    axis=1,
)

output["pred_P_feasible"] = np.round(
    accepted_joint_probability[selected_indices],
    4,
)
output["pred_weakest_constraint"] = [
    constraint_columns[index]
    for index in weakest_constraint_indices
]
output["pred_weakest_prob"] = np.round(
    selected_constraint_probabilities[
        np.arange(len(selected_indices)),
        weakest_constraint_indices,
    ],
    4,
)
output["strategy"] = "graft"
output["source_seed_row"] = accepted_seed_indices[selected_indices]
output["Design_Type"] = "Candidate"

output.to_csv(OUTPUT_PATH, index=False)

# Report score, diversity, and parameter-range coverage for inspection.
selected_values = accepted_values[selected_indices]
known_feasible_standardized = (X[is_feasible] - X.mean(axis=0)) / parameter_std
varying_parameters = dataset_range > 0

selected_range_coverage = np.mean(
    (selected_values.max(axis=0) - selected_values.min(axis=0))[varying_parameters]
    / dataset_range[varying_parameters]
)

print(f"Wrote {len(output):,} candidates to {OUTPUT_PATH}.")
print(
    "Selected P(all pass): "
    f"median={np.median(accepted_joint_probability[selected_indices]):.3f}"
)
print(
    "Candidates inside the known-feasible score band: "
    f"{(accepted_joint_probability[selected_indices] >= np.percentile(known_feasible_score, 10)).sum():,} "
    f"of {len(selected_indices):,}"
)
print(
    "Median standardized pairwise distance: "
    f"candidates={np.median(pdist(standardized_candidates[selected_indices][:500])):.1f}, "
    f"known feasible={np.median(pdist(known_feasible_standardized)):.1f}"
)
print(f"Mean parameter-range coverage: {selected_range_coverage:.2f}")

display(output.head())